<p class="lead">Frequentist statistics answers "how surprising would these data be if nothing were going on?". Bayesian statistics answers the question researchers actually ask — "given these data, what should I now believe?" — at the price of having to say what you believed beforehand. This unit is the one place in the course where that price is paid explicitly, and it is worth paying whenever the data are small, nested, or arriving one experiment at a time: a thesis project with three replicas, a compound measured in two assays, a screening campaign whose hit rate must be estimated after forty tested molecules. We build the machinery from arithmetic upwards — Bayes' theorem on natural frequencies, conjugate updating by hand, grid approximation, and a Markov chain Monte Carlo sampler written from scratch in fifteen lines — and apply it to two problems the laboratory meets constantly: <strong>what is the probability that a predicted hit is really active</strong>, and <strong>how should potencies measured in 108 different assays be combined</strong>. The second question introduces partial pooling, the single most useful idea in this unit, and the R edition cross-checks the hand-written sampler against <code>lme4</code>'s mixed model.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad (optativa, recomendada para proyectos con datos escasos o con estructura jerárquica) presenta el razonamiento bayesiano. Se enuncia el teorema de Bayes en palabras y se aplica a la falacia de la tasa base: dada una prevalencia y un factor de enriquecimiento, ¿cuál es la probabilidad de que un "hit" computacional sea realmente activo? Se realiza la actualización conjugada beta-binomial a mano, la aproximación por malla y un muestreador de Metropolis escrito desde cero, con diagnósticos de convergencia (trazas, R-hat, tamaño efectivo de muestra). Se ajusta un modelo jerárquico de la potencia frente a ROCK1 en 108 ensayos que muestra el encogimiento (shrinkage) de las medias de los ensayos pequeños hacia la media global, y se contrasta con el ajuste de efectos mixtos de <code>lme4::lmer</code>. Se realizan comprobaciones predictivas previas y posteriores, y se comparan intervalos creíbles e intervalos de confianza sobre los mismos datos. <b>Nota:</b> no se usa ningún lenguaje de programación probabilística; todos los modelos se implementan explícitamente, que es el objetivo pedagógico.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>State Bayes' theorem in words and apply it to diagnostic-test and virtual-screening scenarios, including the base-rate fallacy.</li>
<li>Build and interpret simple Bayesian models — beta–binomial, normal with unknown mean — and read a posterior and a credible interval.</li>
<li>Compute a posterior three ways: conjugate algebra, grid approximation and Markov chain Monte Carlo.</li>
<li>Write a Metropolis sampler from scratch and diagnose it with trace plots, acceptance rate, R-hat and effective sample size.</li>
<li>Explain hierarchical (partial-pooling) models, why they suit compounds nested in assays or frames nested in replicas, and what shrinkage does.</li>
<li>Run prior and posterior predictive checks and justify a prior to a referee.</li>
<li>State the difference between a credible interval and a confidence interval, and when the two coincide.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read linearly, running each chunk; then change a prior and re-run, which is the fastest way to develop intuition for how much the prior actually matters. Everything here runs on base R's <code>stats</code><sup class="c"><a href="#r1">[1]</a></sup> plus <code>data.table</code>, <code>ggplot2</code>, <code>Matrix</code> for the frozen fingerprints and <code>lme4</code><sup class="c"><a href="#r2">[2]</a></sup> for the mixed-model cross-check of §9; in a fresh Google Colab R runtime run once: <code>install.packages(c("data.table","ggplot2","Matrix","lme4","R.utils"))</code>. <b>No probabilistic-programming library is required or used:</b> every model is implemented explicitly, which is the teaching point. This unit is marked <em>extended</em>: it assumes <a href="../U7_Estimation/index.html">Unit 7</a> and <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a>, and it pairs with <a href="../U9_Linear_Models/index.html">Unit 9</a>, whose mixed-effects model is the frequentist sibling of §7.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(Matrix); library(lme4)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_bw(base_size = 9) +
          theme(panel.grid.minor = element_blank(),
                plot.title = element_text(face = "bold", size = 10)))
D <- function(f) file.path("..", "data", f)
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| ggplot2", as.character(packageVersion("ggplot2")),
    "| lme4", as.character(packageVersion("lme4")), "\n")



## 1. Bayes' theorem in words

For a hypothesis \( H \) and data \( D \),

\[ \underbrace{P(H\mid D)}_{\text{posterior}} = \frac{\overbrace{P(D\mid H)}^{\text{likelihood}}\;\overbrace{P(H)}^{\text{prior}}}{\underbrace{P(D)}_{\text{evidence}}}, \]

which in words is: **what you should believe after seeing the data is what the data say about each hypothesis, weighted by what you believed before, renormalised**. The denominator is just the sum of the numerator over all hypotheses, so it rarely needs separate attention.

Three consequences are worth internalising before any computation:

1. **The likelihood alone is not enough.** \( P(D\mid H) \) and \( P(H\mid D) \) are different quantities, and confusing them is the formal content of the base-rate fallacy (§2) and of the commonest misreading of a p-value<sup class="c"><a href="#r3">[3]</a></sup><sup class="c"><a href="#r4">[4]</a></sup>.
2. **The prior is an assumption, like every other modelling choice**, and it must be stated, justified and varied in a sensitivity analysis (§5). A model with no prior is not assumption-free; it has made a hidden one.
3. **The output is a distribution, not a number.** Everything a Bayesian analysis reports — a point estimate, an interval, a probability that one series is more potent than another — is a summary of that distribution.

<div class="box bridge"><h4>Domain context — virtual screening, prevalence and enrichment</h4>
<p>A <strong>virtual screen</strong> ranks a large library of purchasable molecules with a computational score and sends the top fraction — typically the best 0.1 % to 1 % — for experimental testing. Two numbers describe how well it worked. The <strong>prevalence</strong> (or base rate) is the proportion of genuinely active molecules in the library; for a real vendor catalogue screened against one target it is of the order of 0.01 % to 1 %. The <strong>enrichment factor</strong> at x %, EF<sub>x%</sub>, is the hit rate among the selected top x % divided by the prevalence: an EF of 10 means the screen found ten times more actives than picking at random would have<sup class="c"><a href="#r5">[5]</a></sup>. Note the ceiling: EF cannot exceed 1/prevalence, since the hit rate cannot exceed 1. The <strong>decoys</strong> used to measure enrichment in a benchmark are molecules presumed inactive, and how they were chosen changes every number<sup class="c"><a href="#r6">[6]</a></sup> — the subject of Unit 19. The quantity a medicinal chemist actually needs is none of these: it is the probability that a molecule the computer selected will turn out to be active, which is a posterior probability and is computed below.</p></div>

## 2. The base-rate fallacy, with natural frequencies

The classic demonstration is medical. A disease affects 1 in 1,000 people. A test detects 99 % of cases and gives a false positive in 5 % of healthy people. Your test is positive. What is the probability that you are ill?

The intuitive answer is "about 95 %". The correct answer is obtained by counting people rather than manipulating probabilities<sup class="c"><a href="#r7">[7]</a></sup>:

| | ill | healthy | total |
|---|---|---|---|
| **test positive** | 99 | 4,995 | 5,094 |
| **test negative** | 1 | 94,905 | 94,906 |
| **total** | 100 | 99,900 | 100,000 |

Of 100,000 people, 100 are ill; the test finds 99 of them. Of the 99,900 healthy people, 5 % — 4,995 — also test positive. So among the 5,094 positive tests only 99 are genuinely ill: **P(ill | positive) = 99/5,094 ≈ 1.9 %**. The test is excellent and the posterior is still under 2 %, because the prior was 0.1 %. Physicians asked this question answer it wrongly at a famously high rate<sup class="c"><a href="#r8">[8]</a></sup>.

The virtual-screening problem has exactly the same arithmetic, with prevalence in place of disease frequency and the enrichment factor in place of sensitivity and specificity. If selecting the top x % of a library enriches actives by a factor EF, then by definition

\[ P(\text{active}\mid\text{selected}) = \mathrm{EF}_{x\%}\times \text{prevalence}. \]



In [ ]:
N <- 100000; prev_med <- 0.001; sens <- 0.99; fpr <- 0.05
ill <- N * prev_med
tab <- matrix(c(ill * sens, ill * (1 - sens), (N - ill) * fpr, (N - ill) * (1 - fpr)), nrow = 2,
              dimnames = list(c("test positive", "test negative"), c("ill", "healthy")))
tab <- cbind(tab, total = rowSums(tab))
print(tab)
cat(sprintf("\nP(ill | positive) = %d / %d = %.4f  (%.1f %%)\n\n",
            tab[1, "ill"], tab[1, "total"], tab[1, "ill"] / tab[1, "total"],
            100 * tab[1, "ill"] / tab[1, "total"]))
prevs <- 10^seq(-4, -0.3, length.out = 200)
curves <- rbindlist(lapply(c(3, 10, 30, 100), function(ef)
  data.table(ef = factor(sprintf("EF = %d", ef), levels = sprintf("EF = %d", c(3, 10, 30, 100))),
             prevalence = prevs, p_active = pmin(1, ef * prevs))))
p <- ggplot(curves, aes(prevalence, p_active, colour = ef)) +
  geom_line(linewidth = 0.7) + scale_x_log10() +
  scale_colour_manual(values = unname(PAL), name = NULL) +
  labs(x = "prevalence of actives in the library (log scale)", y = "P(active | selected by the screen)",
       title = "Even a tenfold enrichment leaves most picks inactive")
print(p)
scen <- data.table(scenario = c("focused library, strong model", "curated subset", "vendor catalogue",
                                "vendor catalogue, excellent model"),
                   prevalence = c(0.20, 0.01, 0.001, 0.001), EF = c(3, 10, 10, 100))
scen[, `P(active | selected)` := pmin(1, prevalence * EF)]
scen[, `actives among 100 ordered` := round(100 * `P(active | selected)`, 1)]
knitr::kable(scen, digits = 4)



<p class="caption"><b>Figure 1.</b> Posterior probability that a selected molecule is active, as a function of the library's prevalence of actives, for four enrichment factors. The curves are straight lines on these axes because the posterior is simply EF × prevalence until it saturates at 1. At a realistic catalogue prevalence of 0.1 %, even an enrichment factor of 10 gives a 1 % chance that any individual pick is active — one hit per hundred compounds ordered — and an enrichment of 100, which would be an outstanding result, gives one in ten. The table below the figure converts the same arithmetic into compounds ordered.</p>

This single calculation explains several things students find puzzling. It explains why experimental groups ask for 50–100 compounds from a virtual screen rather than 5: at a 1–10 % expected hit rate, ordering five compounds has a substantial chance of returning nothing. It explains why benchmark enrichment factors are optimistic: a benchmark set has a prevalence of several per cent by construction (ours is 10.9 %), a hundred times higher than a real catalogue. And it is structurally the same calculation as the positive predictive value of a significant p-value<sup class="c"><a href="#r4">[4]</a></sup><sup class="c"><a href="#r3">[3]</a></sup>: a "significant" result from a field in which few hypotheses are true is mostly a false positive, for exactly the reason the medical table above is mostly false positives.

<div class="box warn"><h4>Substitution in the R edition — fingerprints</h4>
<p>The ligand-based score below is the maximum Tanimoto similarity of each library molecule to 25 reference actives. R has no RDKit binding in this environment, so the Morgan fingerprints (radius 2, 2,048 bits) are loaded from the frozen sparse file <code>data/screening_morgan_sparse.csv.gz</code> and rebuilt with <code>Matrix::sparseMatrix()</code>, exactly as in <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a>; the provenance is in <a href="../../data/manifest_fingerprints.json">data/manifest_fingerprints.json</a> and the structures ship as <code>data/rock1_curated.sdf.gz</code> for anyone who wants to recompute them with ChemmineR.</p></div>



In [ ]:
lib <- fread(D("screening_library.csv"))
fp  <- fread(D("screening_morgan_sparse.csv.gz"))
M   <- sparseMatrix(i = match(fp$id, lib$inchikey), j = fp$bit + 1L, x = 1, dims = c(nrow(lib), 2048))
pc  <- rowSums(M)
set.seed(0)
ref <- sample(which(lib$label == 1), 25)
inter <- as.matrix(M %*% t(M[ref, , drop = FALSE]))
score <- apply(inter / (outer(pc, pc[ref], "+") - inter), 1, max)
keep <- rep(TRUE, nrow(lib)); keep[ref] <- FALSE
s_lib <- score[keep]; y_lib <- lib$label[keep]
prev_bench <- mean(y_lib)
cat(sprintf("benchmark prevalence of actives: %.3f -> maximum possible EF = 1/prevalence = %.1f\n",
            prev_bench, 1 / prev_bench))
for (frac in c(0.01, 0.05, 0.10, 0.20)) {
  top <- order(-s_lib)[1:floor(frac * length(s_lib))]
  cat(sprintf("  top %5.1f %% : hit rate %.3f -> EF = %.2f\n",
              100 * frac, mean(y_lib[top]), mean(y_lib[top]) / prev_bench))
}
ef10 <- mean(y_lib[order(-s_lib)[1:floor(0.10 * length(s_lib))]]) / prev_bench
for (pr in c(0.10, 0.01, 0.001))
  cat(sprintf("if the real library had prevalence %.3f, this EF of %.2f gives P(active|pick) = %.4f\n",
              pr, ef10, min(1, ef10 * pr)))



The ranking is genuinely good — at the top 1 % and even the top 5 % every selected molecule is an active, which is the maximum the benchmark allows, since the enrichment factor is capped at 1/prevalence = 9.2. That cap is the first thing to check when reading an enrichment number in a paper: an EF of 9.2 on a benchmark with 10.9 % actives is a *perfect* result, while the same screen applied to a catalogue with 0.1 % actives would still return 99 inactive compounds in every 100 ordered (P(active | pick) = 0.008 at EF 8.0).

## 3. The beta–binomial, by hand

The simplest real Bayesian model: we test \( n \) compounds from a screen, \( k \) turn out active, and we want the posterior for the hit rate \( \theta \).

- **Likelihood**: \( P(k \mid \theta) \propto \theta^{k}(1-\theta)^{n-k} \) — the binomial.
- **Prior**: \( \theta \sim \text{Beta}(a, b) \), whose density is proportional to \( \theta^{a-1}(1-\theta)^{b-1} \). Beta(1,1) is uniform; Beta(2,18) has mean 0.1 and is the sort of weakly informative prior a chemist's experience justifies.
- **Posterior**: multiplying, \( \theta \mid k \sim \text{Beta}(a+k,\; b+n-k) \). The prior is **conjugate**: the posterior is in the same family, and updating is addition.

### Worked example

Prior Beta(2, 18) — "we expect about 10 % hits, with the confidence of about 20 prior observations". We test **40** compounds and **6** are active.

Posterior = Beta(2 + 6, 18 + 40 − 6) = **Beta(8, 52)**.

| quantity | formula | value |
|---|---|---|
| prior mean | a/(a+b) = 2/20 | 0.1000 |
| maximum-likelihood estimate | k/n = 6/40 | 0.1500 |
| posterior mean | 8/(8+52) | **0.1333** |
| posterior mode | (8−1)/(8+52−2) = 7/58 | 0.1207 |
| posterior SD | √(ab/((a+b)²(a+b+1))) with a=8, b=52 | 0.0435 |

The posterior mean sits between the prior mean and the data estimate, closer to the data because 40 observations outweigh the 20 observations' worth of prior. That compromise is the whole of Bayesian updating, and it is also the mechanism behind the shrinkage of §7.



In [ ]:
a0 <- 2; b0 <- 18; n_tested <- 40; k_active <- 6
a1 <- a0 + k_active; b1 <- b0 + n_tested - k_active
cat(sprintf("prior Beta(%d, %d): mean %.4f\n", a0, b0, a0 / (a0 + b0)))
cat(sprintf("data: %d active out of %d -> MLE %.4f\n", k_active, n_tested, k_active / n_tested))
cat(sprintf("posterior Beta(%d, %d): mean %.4f, mode %.4f, sd %.4f\n", a1, b1, a1 / (a1 + b1),
            (a1 - 1) / (a1 + b1 - 2), sqrt(a1 * b1 / ((a1 + b1)^2 * (a1 + b1 + 1)))))
cat(sprintf("95 %% credible interval (equal-tailed): [%.4f, %.4f]\n", qbeta(0.025, a1, b1), qbeta(0.975, a1, b1)))
cat(sprintf("P(hit rate > 0.10 | data) = %.3f\n", pbeta(0.10, a1, b1, lower.tail = FALSE)))
cat(sprintf("P(hit rate > 0.20 | data) = %.3f\n", pbeta(0.20, a1, b1, lower.tail = FALSE)))
th <- seq(0, 0.6, length.out = 500)
lik <- dbinom(k_active, n_tested, th); lik <- lik / (sum(lik) * diff(th)[1])
dens <- rbind(data.table(theta = th, d = dbeta(th, a0, b0), what = "prior Beta(2, 18)"),
              data.table(theta = th, d = lik, what = "likelihood (6 of 40), normalised"),
              data.table(theta = th, d = dbeta(th, a1, b1), what = "posterior Beta(8, 52)"))
band <- data.table(theta = th[th >= qbeta(0.025, a1, b1) & th <= qbeta(0.975, a1, b1)])
band[, d := dbeta(theta, a1, b1)]
p <- ggplot(dens, aes(theta, d, colour = what)) +
  geom_area(data = band, aes(theta, d), inherit.aes = FALSE, fill = PAL["orange"], alpha = 0.2) +
  geom_line(linewidth = 0.8) +
  scale_colour_manual(values = unname(PAL[c("green", "blue", "orange")]), name = NULL) +
  labs(x = "hit rate theta", y = "density",
       title = "Prior x likelihood = posterior, with the 95 % credible interval shaded")
print(p)



<p class="caption"><b>Figure 2.</b> The three densities of a Bayesian update. The prior (blue) is broad and centred at 0.10; the normalised likelihood (green) is centred at the data estimate 0.15 and is wider still, because 40 observations do not pin down a proportion tightly; the posterior (orange) is narrower than either and sits at 0.1333, between them. The shaded region is the 95 % credible interval, 0.060 to 0.229. Note what the posterior lets us say directly: the probability that the true hit rate exceeds 10 % is 0.766, and that it exceeds 20 % is 0.075 — statements about the parameter, which a confidence interval is not allowed to make (§10).</p>



In [ ]:
true_rate <- 0.12
set.seed(0)
outcomes <- runif(400) < true_rate
seqs <- rbindlist(lapply(c(0, 10, 40, 120, 400), function(n_seen) {
  k <- sum(outcomes[seq_len(n_seen)])
  data.table(theta = th, d = dbeta(th, a0 + k, b0 + n_seen - k),
             what = sprintf("after %d tested (%d active)", n_seen, k))
}))
seqs[, what := factor(what, levels = unique(what))]
ns <- 1:400; ks <- cumsum(outcomes)
run <- data.table(n = ns, mean = (a0 + ks) / (a0 + b0 + ns), mle = ks / ns,
                  lo = qbeta(0.025, a0 + ks, b0 + ns - ks), hi = qbeta(0.975, a0 + ks, b0 + ns - ks))
p1 <- ggplot(seqs, aes(theta, d, colour = what)) + geom_line(linewidth = 0.6) +
  geom_vline(xintercept = true_rate, linetype = 2) +
  coord_cartesian(xlim = c(0, 0.4)) +
  scale_colour_manual(values = c("grey70", unname(PAL)), name = NULL) +
  theme(legend.position = "bottom", legend.text = element_text(size = 6), legend.key.size = unit(7, "pt")) +
  labs(x = "hit rate theta", y = "density", title = "the posterior concentrates")
p2 <- ggplot(run, aes(n)) +
  geom_ribbon(aes(ymin = lo, ymax = hi), fill = PAL["blue"], alpha = 0.2) +
  geom_line(aes(y = mean), colour = PAL["blue"]) +
  geom_line(aes(y = mle), colour = PAL["green"], linewidth = 0.3) +
  geom_hline(yintercept = true_rate, linetype = 2) +
  coord_cartesian(ylim = c(0, 0.35)) +
  labs(x = "compounds tested", y = "hit rate",
       title = "the prior washes out (band: 95 % CrI; green: k/n)")
print(p1); print(p2)
for (n_seen in c(5, 20, 100, 400)) {
  k <- sum(outcomes[1:n_seen])
  cat(sprintf("after %3d tested, %3d active: MLE %.3f | posterior mean %.3f | 95%% CrI [%.3f, %.3f]\n",
              n_seen, k, k / n_seen, (a0 + k) / (a0 + b0 + n_seen),
              qbeta(0.025, a0 + k, b0 + n_seen - k), qbeta(0.975, a0 + k, b0 + n_seen - k)))
}



<p class="caption"><b>Figure 3.</b> Left: the posterior after 0, 10, 40, 120 and 400 tested compounds, with the true rate of 0.12 dashed. Right: the posterior mean with its credible band against the running maximum-likelihood estimate (green). Two features matter. The maximum-likelihood estimate is wild at small n — after five compounds it is exactly 0 here, which no one believes — while the posterior mean is pulled towards the prior and gives 0.080 with an interval of 0.010 to 0.211. And the prior's influence decays: after 400 compounds (37 active) the posterior mean is 0.093 against a maximum-likelihood estimate of 0.092, indistinguishable, because 400 observations swamp 20 observations' worth of prior belief.</p>

This is the general answer to "but isn't the prior arbitrary?". A prior worth 20 observations matters when you have 5 and is irrelevant when you have 400. The honest practice is to say how much information your prior carries, and to check (§5) that the conclusion does not depend on it.

## 4. Grid approximation: Bayes without algebra

Conjugate priors are convenient and rare. The general method that always works in one or two dimensions is **grid approximation**: lay a grid over the parameter, evaluate prior × likelihood at every grid point, and normalise. It is the numerical definition of Bayes' theorem, and it makes the mathematics unnecessary.



In [ ]:
grid <- seq(1e-4, 1 - 1e-4, length.out = 1000)
dx <- diff(grid)[1]
lp <- dbeta(grid, a0, b0, log = TRUE) + dbinom(k_active, n_tested, grid, log = TRUE)
w <- exp(lp - max(lp)); w <- w / (sum(w) * dx)
cdf <- cumsum(w) * dx
cat(sprintf("grid vs analytic posterior mean: %.5f vs %.5f\n", sum(grid * w) * dx, a1 / (a1 + b1)))
cat(sprintf("grid vs analytic 95%% interval : [%.4f, %.4f] vs [%.4f, %.4f]\n",
            grid[which.min(abs(cdf - 0.025))], grid[which.min(abs(cdf - 0.975))],
            qbeta(0.025, a1, b1), qbeta(0.975, a1, b1)))
mix <- 0.7 * dbeta(grid, 2, 60) + 0.3 * dbeta(grid, 12, 20)      ## no conjugate partner exists
lp2 <- log(mix) + dbinom(k_active, n_tested, grid, log = TRUE)
w2 <- exp(lp2 - max(lp2)); w2 <- w2 / (sum(w2) * dx)
dd <- rbind(data.table(theta = grid, d = mix / (sum(mix) * dx), what = "bimodal prior (no conjugate form)"),
            data.table(theta = grid, d = w2, what = "posterior by grid approximation"),
            data.table(theta = grid, d = w, what = "posterior under the Beta(2,18) prior"))
p <- ggplot(dd, aes(theta, d, colour = what)) + geom_line(linewidth = 0.7) +
  coord_cartesian(xlim = c(0, 0.6)) +
  scale_colour_manual(values = unname(PAL[c("green", "orange", "blue")]), name = NULL) +
  theme(legend.position = "bottom", legend.text = element_text(size = 6.5)) +
  labs(x = "hit rate theta", y = "density", title = "Grid approximation works for any prior you can write down")
print(p)
cat(sprintf("bimodal-prior posterior mean %.4f (vs %.4f under the conjugate prior)\n",
            sum(grid * w2) * dx, a1 / (a1 + b1)))
cat(sprintf("grid points: %d | cost: %d likelihood evaluations\n", length(grid), length(grid)))



<p class="caption"><b>Figure 4.</b> Grid approximation. The blue curve reproduces the conjugate posterior of Figure 2 — the grid mean is 0.13333 against the analytic 0.13333, and the interval matches to three decimals — using nothing but 1,000 evaluations of prior × likelihood. The orange curve uses a deliberately awkward bimodal prior (green), a chemist who believes the campaign is either a dud or quite good, for which no conjugate update exists; the posterior is computed just as easily and ends up at 0.1409, slightly higher than the conjugate one.</p>

Grid approximation is exact up to the grid resolution and completely transparent, which is why it is the right first tool. Its fatal limitation is dimensionality: a grid of 1,000 points per parameter costs 10<sup>3</sup> evaluations for one parameter, 10<sup>6</sup> for two and 10<sup>9</sup> for three. Beyond two or three parameters it is hopeless, and that is precisely the gap Markov chain Monte Carlo fills (§6).

## 5. How much does the prior matter?

The question a referee will ask is "would a different prior have changed your conclusion?". The answer is obtained by refitting under several priors and tabulating — a **prior sensitivity analysis**, which costs three lines and should appear in every Bayesian paper.



In [ ]:
priors <- list("uniform Beta(1,1)" = c(1, 1), "Jeffreys Beta(0.5,0.5)" = c(0.5, 0.5),
               "weak, 10 % Beta(2,18)" = c(2, 18), "strong, 10 % Beta(10,90)" = c(10, 90),
               "strong and wrong, 50 % Beta(50,50)" = c(50, 50))
k400 <- sum(outcomes)
sens <- rbindlist(lapply(names(priors), function(nm) {
  pp <- priors[[nm]]
  rbindlist(lapply(list(c(40, k_active), c(400, k400)), function(z) {
    n_ <- z[1]; k_ <- z[2]
    data.table(prior = nm, n_tested = n_, mean = (pp[1] + k_) / (sum(pp) + n_),
               lo = qbeta(0.025, pp[1] + k_, pp[2] + n_ - k_),
               hi = qbeta(0.975, pp[1] + k_, pp[2] + n_ - k_))
  }))
}))
sens[, prior := factor(prior, levels = rev(names(priors)))]
p <- ggplot(sens, aes(mean, prior, colour = factor(n_tested))) +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.2, position = position_dodge(width = 0.5)) +
  geom_point(size = 1.7, position = position_dodge(width = 0.5)) +
  geom_vline(xintercept = true_rate, linetype = 2) +
  scale_colour_manual(values = unname(PAL[c("orange", "blue")]), name = "compounds tested") +
  labs(x = "posterior mean and 95 % credible interval for the hit rate", y = NULL,
       title = "Prior sensitivity: the data win if there are enough of them")
print(p)
cat("range of posterior means across the five priors:\n")
cat(sprintf("  with n = 40  : %.3f to %.3f  (spread %.3f)\n",
            min(sens[n_tested == 40]$mean), max(sens[n_tested == 40]$mean),
            diff(range(sens[n_tested == 40]$mean))))
cat(sprintf("  with n = 400 : %.3f to %.3f  (spread %.3f)\n",
            min(sens[n_tested == 400]$mean), max(sens[n_tested == 400]$mean),
            diff(range(sens[n_tested == 400]$mean))))
knitr::kable(sens, digits = 3)



<p class="caption"><b>Figure 5.</b> Posterior means and credible intervals for five priors at two sample sizes (dashed line: the true rate of 0.12). With 40 compounds tested the five answers span 0.287 in the hit rate — the deliberately wrong Beta(50,50) prior drags the estimate to 0.400 against a maximum-likelihood value of 0.15 — and a reader is entitled to ask which prior was used. With 400 compounds the four defensible priors collapse onto 0.093–0.095, a spread of 0.002 among them, and only the strong, wrong prior remains visibly displaced at 0.174. The rule: <strong>report the sensitivity table, and if the conclusion changes across defensible priors, the data are not sufficient to support it</strong>.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>A collaborator insists on a Beta(1, 1) uniform prior "to avoid bias". Compute the posterior for a campaign in which <b>0 of 12</b> compounds were active, under Beta(1,1), Beta(2,18) and Beta(0.5,0.5). Which prior gives the most defensible answer, and what does each imply about the probability that the true hit rate exceeds 20 %? (Note that the maximum-likelihood estimate here is exactly 0, with a Wald confidence interval that R's own arithmetic reports as [0, 0] — see §10.)</p></div>

## 6. Markov chain Monte Carlo, written from scratch

When the parameter space has more than two or three dimensions, we can no longer evaluate the posterior everywhere. **Markov chain Monte Carlo** (MCMC) instead constructs a random walk whose long-run visiting frequency *is* the posterior, so a histogram of the walk approximates the distribution. The **Metropolis algorithm**<sup class="c"><a href="#r9">[9]</a></sup><sup class="c"><a href="#r10">[10]</a></sup> is four lines:

1. from the current position \( \theta \), propose \( \theta' = \theta + \text{noise} \);
2. compute the ratio \( r = p(\theta'\mid D)/p(\theta\mid D) \) — note that the unknown normalising constant cancels, which is the trick that makes the whole thing possible;
3. accept \( \theta' \) with probability \( \min(1, r) \), otherwise stay at \( \theta \);
4. repeat, recording every position.

Modern samplers — Gibbs<sup class="c"><a href="#r11">[11]</a></sup>, Hamiltonian Monte Carlo<sup class="c"><a href="#r12">[12]</a></sup> and the No-U-Turn sampler used by Stan and its R interfaces — are smarter about step 1, but the logic is identical.



In [ ]:
log_posterior <- function(theta, a = a0, b = b0, n = n_tested, k = k_active) {
  if (theta <= 0 || theta >= 1) return(-Inf)
  dbeta(theta, a, b, log = TRUE) + dbinom(k, n, theta, log = TRUE)
}
metropolis <- function(log_p, start, step, n_iter, seed) {
  set.seed(seed)
  chain <- numeric(n_iter); theta <- start; lp <- log_p(theta); accepted <- 0
  for (i in seq_len(n_iter)) {
    prop <- theta + rnorm(1, 0, step)
    lp_prop <- log_p(prop)
    if (log(runif(1)) < lp_prop - lp) { theta <- prop; lp <- lp_prop; accepted <- accepted + 1 }
    chain[i] <- theta
  }
  list(chain = chain, acceptance = accepted / n_iter)
}
chains <- lapply(1:4, function(i)
  metropolis(log_posterior, start = c(0.02, 0.10, 0.25, 0.45)[i], step = 0.15, n_iter = 8000, seed = i))
burn <- 1000
post_chains <- lapply(chains, function(z) z$chain[(burn + 1):8000])
pooled <- unlist(post_chains)
cat(sprintf("acceptance rates: %s  (0.2-0.5 is the healthy range for random-walk Metropolis)\n",
            paste(sprintf("%.3f", sapply(chains, `[[`, "acceptance")), collapse = ", ")))
cat(sprintf("posterior mean  : MCMC %.4f vs analytic %.4f\n", mean(pooled), a1 / (a1 + b1)))
cat(sprintf("95 %% interval   : MCMC [%.4f, %.4f] vs analytic [%.4f, %.4f]\n",
            quantile(pooled, 0.025), quantile(pooled, 0.975), qbeta(0.025, a1, b1), qbeta(0.975, a1, b1)))


In [ ]:
rhat <- function(chs) {
  m <- do.call(rbind, chs); n <- ncol(m)
  W <- mean(apply(m, 1, var)); B <- n * var(rowMeans(m))
  sqrt(((n - 1) / n * W + B / n) / W)
}
ess <- function(x) {
  n <- length(x)
  ac <- as.vector(acf(x, lag.max = min(1000, n - 1), plot = FALSE)$acf)
  stop_at <- which(ac < 0.05)[1]; if (is.na(stop_at)) stop_at <- length(ac)
  n / (1 + 2 * sum(ac[2:(stop_at - 1)]))
}
op <- par(mfrow = c(1, 3), mar = c(4, 4, 2.6, 1), cex = 0.72)
plot(NA, xlim = c(0, 400), ylim = c(0, 0.5), xlab = "iteration (first 400)", ylab = "theta",
     main = "four chains find the posterior")
for (i in 1:4) lines(chains[[i]]$chain[1:400], col = unname(PAL)[i], lwd = 0.7)
abline(v = burn / 10, lty = 2)
plot(NA, xlim = c(burn, 8000), ylim = c(0, 0.35), xlab = "iteration (after burn-in)", ylab = "theta",
     main = "a healthy trace is a fuzzy caterpillar")
for (i in 1:4) lines((burn + 1):8000, post_chains[[i]], col = adjustcolor(unname(PAL)[i], 0.6), lwd = 0.25)
hist(pooled, breaks = 60, freq = FALSE, col = "grey80", border = NA, xlim = c(0, 0.4),
     xlab = "theta", main = "MCMC and analytic agree")
curve(dbeta(x, a1, b1), add = TRUE, col = PAL["orange"], lwd = 1.8)
par(op)
cat(sprintf("R-hat (should be < 1.01): %.4f\n", rhat(post_chains)))
cat(sprintf("effective sample size per chain: %s  (of %d draws each)\n",
            paste(round(sapply(post_chains, ess)), collapse = ", "), length(post_chains[[1]])))
cat(sprintf("Monte-Carlo standard error of the posterior mean: %.5f\n",
            sd(pooled) / sqrt(sum(sapply(post_chains, ess)))))



<p class="caption"><b>Figure 6.</b> Diagnosing a sampler. Left: the first 400 iterations of four chains started deliberately far apart at 0.02, 0.10, 0.25 and 0.45; all four reach the same region within roughly a hundred steps, and everything before the dashed line is discarded as burn-in. Centre: after burn-in the traces look like overlapping fuzzy caterpillars with no trends and no sticking — the visual signature of a healthy chain. Right: the pooled histogram lies under the analytic posterior, which is the strongest possible check since here we know the right answer: MCMC mean 0.1336 against the analytic 0.1333, interval [0.060, 0.230] against [0.060, 0.229].</p>

Three numbers must accompany any MCMC result. The **acceptance rate** should be roughly 0.2–0.5 for a random-walk sampler (ours is 0.32–0.34); much lower means the steps are too large and the chain sticks, much higher means they are too small and it crawls. **R-hat** compares the variance between chains with the variance within them and should be below about 1.01<sup class="c"><a href="#r13">[13]</a></sup><sup class="c"><a href="#r14">[14]</a></sup>; ours is 1.0001. The **effective sample size** counts how many independent draws the autocorrelated chain is worth — 1,200 to 1,500 of 7,000 draws per chain here, which is exactly the quantity Unit 7 computed for molecular-dynamics frames, and for the same reason. Reporting a posterior without these is like reporting a simulation average without saying how long the trajectory was.



In [ ]:
op <- par(mfrow = c(1, 3), mar = c(4, 4, 2.6, 1), cex = 0.72)
step_rows <- rbindlist(lapply(list(c(0.005, 1), c(0.15, 2), c(1.00, 3)), function(z) {
  step <- z[1]
  cc <- lapply(1:4, function(i) metropolis(log_posterior, 0.15, step, 4000, seed = 100 + i))
  plot(NA, xlim = c(0, 2000), ylim = c(0, 0.35), xlab = "iteration",
       ylab = if (z[2] == 1) "theta" else "",
       main = sprintf("step = %.3f (%s)", step, c("too small", "about right", "too large")[z[2]]))
  for (i in 1:4) lines(cc[[i]]$chain[1001:3000], col = adjustcolor(unname(PAL)[i], 0.8), lwd = 0.35)
  pcs <- lapply(cc, function(z2) z2$chain[1001:4000])
  data.table(step = step, acceptance = mean(sapply(cc, `[[`, "acceptance")), R_hat = rhat(pcs),
             ESS_per_chain = mean(sapply(pcs, ess)), posterior_mean = mean(unlist(pcs)))
}))
par(op)
cat(sprintf("analytic posterior mean %.4f\n\n", a1 / (a1 + b1)))
knitr::kable(step_rows, digits = 4)



<p class="caption"><b>Figure 7.</b> The same sampler at three step sizes. With steps that are too small (left) the chain accepts 97 % of proposals and drifts slowly, so the trace is a smooth wandering line rather than a caterpillar; with steps that are too large (right) only 5.4 % of proposals are accepted and the trace is a staircase of long flat sections. The middle panel is what a healthy chain looks like. The table gives the numbers, and they contain the warning: the posterior mean is within 0.005 of the analytic value in all three cases — which is exactly the danger — while the acceptance rate (0.970, 0.333, 0.054), R-hat (1.060, 1.000, 1.006) and effective sample size (17, 687, 117 per chain) separate them clearly. <strong>MCMC fails silently; the diagnostics are not optional.</strong></p>

## 7. Hierarchical models and shrinkage

Here is the problem this unit exists for. Our ROCK1 data come from many different assays, run in different laboratories with different protocols. Some assays measured hundreds of compounds, some measured three. What is "the" potency level of each assay?

Three possible answers:

- **Complete pooling** — ignore the assay labels, use the grand mean for everything. Throws away real between-assay differences.
- **No pooling** — estimate each assay separately. An assay with three measurements gets an estimate with an enormous standard error, and the most extreme-looking assays will mostly be the small ones.
- **Partial pooling** — model the assay means as draws from a common distribution and let the data decide how much they differ. Each assay's estimate is then a compromise between its own mean and the grand mean, weighted by how much information it carries.

Partial pooling is what a hierarchical model does. Formally, for measurement \( i \) in assay \( j \):

\[ y_{ij}\sim\mathcal{N}(\theta_j,\sigma^2),\qquad \theta_j\sim\mathcal{N}(\mu,\tau^2),\qquad \mu,\sigma,\tau \text{ given weakly informative priors.} \]

The posterior mean of \( \theta_j \) is approximately \( \lambda_j\bar y_j + (1-\lambda_j)\mu \) with \( \lambda_j = \tau^2/(\tau^2+\sigma^2/n_j) \) — large assays keep their own mean, small assays are pulled towards the overall level. That pull is **shrinkage**, and it is not a Bayesian curiosity: it provably improves the average accuracy of the whole set of estimates<sup class="c"><a href="#r15">[15]</a></sup>, and it is the engine of the empirical-Bayes methods used throughout genomics<sup class="c"><a href="#r16">[16]</a></sup><sup class="c"><a href="#r17">[17]</a></sup>.



In [ ]:
raw <- fread(D("rock1_chembl_raw.csv"))
ok <- raw[standard_relation == "=" & !is.na(standard_value) &
          standard_units == "nM" & standard_value > 0]
ok[, pAct := 9 - log10(as.numeric(standard_value))]
ok <- ok[pAct > 2 & pAct < 12]
ok[, n_in_assay := .N, by = assay_chembl_id]
ok <- ok[n_in_assay >= 3]
ok[, code := .GRP, by = assay_chembl_id]
J <- uniqueN(ok$code); yobs <- ok$pAct; codes <- ok$code
nj <- as.numeric(table(codes)); ybar <- as.numeric(tapply(yobs, codes, mean))
assay_ids <- ok[, .(id = assay_chembl_id[1]), by = code][order(code)]$id
cat(sprintf("assays with at least 3 exact measurements : %d\n", J))
cat(sprintf("measurements used                         : %d\n", length(yobs)))
cat(sprintf("assay sizes: min %d, median %d, max %d\n", min(nj), median(nj), max(nj)))
cat(sprintf("grand mean pActivity %.3f | SD of the %d assay means %.3f | pooled within-assay SD %.3f\n",
            mean(yobs), J, sd(ybar), sqrt(sum((yobs - ybar[codes])^2) / (length(yobs) - J))))
cat(sprintf("the three smallest assays have means %s\n",
            paste(sprintf("%.3f", ybar[order(nj)][1:3]), collapse = ", ")))


In [ ]:
hierarchical_sampler <- function(seed, n_iter = 4000, burn = 1000) {
  set.seed(seed)
  mu <- 6.5; log_tau <- log(0.8); log_sig <- log(0.8); theta <- ybar; accepted <- 0
  keep <- matrix(NA_real_, nrow = n_iter - burn, ncol = 3 + J)
  log_p_scales <- function(lt, ls, theta, mu) {
    ## HalfNormal(1) priors on tau and sigma, plus the log-Jacobian of the log transform
    t <- exp(lt); sg <- exp(ls)
    prior <- -0.5 * t^2 - 0.5 * sg^2 + lt + ls
    like  <- -J * lt - 0.5 * sum((theta - mu)^2) / t^2 -
             length(yobs) * ls - 0.5 * sum((yobs - theta[codes])^2) / sg^2
    prior + like
  }
  for (it in seq_len(n_iter)) {
    tau <- exp(log_tau); sig <- exp(log_sig)
    prec  <- nj / sig^2 + 1 / tau^2                                   ## Gibbs step for every theta_j
    theta <- rnorm(J, (nj * ybar / sig^2 + mu / tau^2) / prec, 1 / sqrt(prec))
    mu    <- rnorm(1, mean(theta), tau / sqrt(J))                     ## Gibbs step for mu (flat prior)
    p_lt <- log_tau + rnorm(1, 0, 0.12); p_ls <- log_sig + rnorm(1, 0, 0.05)   ## Metropolis block
    if (log(runif(1)) < log_p_scales(p_lt, p_ls, theta, mu) - log_p_scales(log_tau, log_sig, theta, mu)) {
      log_tau <- p_lt; log_sig <- p_ls; accepted <- accepted + 1
    }
    if (it > burn) keep[it - burn, ] <- c(mu, exp(log_tau), exp(log_sig), theta)
  }
  list(draws = keep, acceptance = accepted / n_iter)
}
draws <- lapply(1:4, hierarchical_sampler)
post_h <- do.call(rbind, lapply(draws, `[[`, "draws"))
mu_post <- post_h[, 1]; tau_post <- post_h[, 2]; sig_post <- post_h[, 3]
theta_post <- post_h[, 4:(3 + J)]
cat(sprintf("acceptance rate of the Metropolis block: %s\n",
            paste(sprintf("%.3f", sapply(draws, `[[`, "acceptance")), collapse = ", ")))
cat(sprintf("R-hat  mu %.4f | tau %.4f | sigma %.4f\n",
            rhat(lapply(draws, function(d) d$draws[, 1])),
            rhat(lapply(draws, function(d) d$draws[, 2])),
            rhat(lapply(draws, function(d) d$draws[, 3]))))
cat(sprintf("mu    (grand mean potency) %.3f  95%% CrI [%.3f, %.3f]\n",
            mean(mu_post), quantile(mu_post, 0.025), quantile(mu_post, 0.975)))
cat(sprintf("tau   (between-assay SD)   %.3f  95%% CrI [%.3f, %.3f]\n",
            mean(tau_post), quantile(tau_post, 0.025), quantile(tau_post, 0.975)))
cat(sprintf("sigma (within-assay SD)    %.3f  95%% CrI [%.3f, %.3f]\n",
            mean(sig_post), quantile(sig_post, 0.025), quantile(sig_post, 0.975)))



The sampler used 3,205 measurements from 108 assays whose sizes run from 3 to 501 compounds, and its Metropolis block accepted about 19 % of proposals — acceptable for a two-dimensional block, and R-hat is 1.002 or better for all three global parameters.

The two variance components are the scientific output of this model, and they answer a question the laboratory cares about. The **within-assay** standard deviation, 0.828 log units, is how much compounds differ inside one assay — real structure–activity variation plus measurement error. The **between-assay** standard deviation, 0.843 log units, is how much assays differ from each other after accounting for that: slightly larger, and the posterior probability that τ exceeds σ is 0.59, so the two sources of variation are of comparable size. That quantifies the warning of <a href="../U7_Estimation/index.html">Unit 7</a> that potencies from different sources are not interchangeable<sup class="c"><a href="#r18">[18]</a></sup><sup class="c"><a href="#r19">[19]</a></sup>. The practical implication for any modelling project is that an assay label is a variable, not a nuisance.



In [ ]:
theta_hat <- colMeans(theta_post)
theta_lo <- apply(theta_post, 2, quantile, 0.025)
theta_hi <- apply(theta_post, 2, quantile, 0.975)
shift <- theta_hat - ybar
sh <- data.table(n = nj, raw = ybar, post = theta_hat, shift = shift)
op <- par(mfrow = c(1, 2), mar = c(4.2, 4.2, 2.6, 1), cex = 0.72)
plot(nj, ybar, log = "x", pch = 19, cex = 0.6, col = PAL["orange"],
     xlab = "measurements in the assay (log scale)", ylab = "assay mean pActivity",
     main = "small assays are pulled towards the centre")
segments(nj, ybar, nj, theta_hat, col = "#99999999")
points(nj, theta_hat, pch = 19, cex = 0.6, col = PAL["blue"])
abline(h = mean(mu_post), lty = 2)
legend("topright", c("raw assay mean", "posterior mean", "grand mean mu"),
       col = c(unname(PAL[c("orange", "blue")]), "black"), pch = c(19, 19, NA), lty = c(NA, NA, 2),
       bty = "n", cex = 0.8)
plot(nj, abs(shift), log = "x", pch = 19, cex = 0.6, col = PAL["green"],
     xlab = "measurements in the assay (log scale)", ylab = "|shift| from the raw mean (log units)",
     main = "shrinkage decays as 1/n")
lam_theory <- mean(tau_post)^2 / (mean(tau_post)^2 + mean(sig_post)^2 / sort(nj))
lines(sort(nj), (1 - lam_theory) * mean(abs(ybar - mean(mu_post))), col = PAL["orange"], lwd = 1.6)
par(op)
cat("assay           n   raw mean   posterior mean    shift    95 % credible interval\n")
for (i in c(order(nj)[1:3], order(-nj)[1:2]))
  cat(sprintf("%-14s %3d   %8.3f   %13.3f   %+6.3f   [%.3f, %.3f]\n",
              assay_ids[i], nj[i], ybar[i], theta_hat[i], shift[i], theta_lo[i], theta_hi[i]))
cat(sprintf("\nmean |shift| for assays with n <= 5 : %.3f log units\n", mean(abs(shift[nj <= 5]))))
cat(sprintf("mean |shift| for assays with n >= 50: %.3f log units\n", mean(abs(shift[nj >= 50]))))
cat(sprintf("largest single shift: %.3f log units\n", max(abs(shift))))



<p class="caption"><b>Figure 8.</b> Left: each assay's raw mean (orange) joined to its partially pooled posterior mean (blue), against the number of measurements. The assays on the left — three or four compounds each — are pulled substantially towards the grand mean of 6.75, by up to 0.466 log units; the assays on the right, with hundreds of measurements, do not move at all (0.001 and 0.002 log units for the two largest). Right: the size of the shift against assay size, with the theoretical (1 − λ) curve in orange; the agreement confirms that the sampler is doing what the algebra says. The mean shift for assays with five or fewer measurements, 0.151 log units, is nineteen times that for assays with fifty or more (0.008).</p>

This figure is the argument for hierarchical modelling in one picture. The assay CHEMBL1099847, with three compounds averaging pActivity 7.967, is probably not a special assay; it is a small sample from the same population, and the model's estimate of 7.667 is a better bet about its next compound than its own mean is. The same logic applies directly to the structures this group works with: **frames nested in replicas** (Unit 20), **compounds nested in series**, **sites nested in patients** (<a href="../U9_Linear_Models/index.html">Unit 9</a>'s blocking example). The frequentist counterpart is the mixed-effects model of §9 below; the Bayesian version additionally gives a full posterior for each group and for the variance components, which is what makes statements like "the probability that assay A is more sensitive than assay B is 0.83" available.

### 7.1 Does partial pooling actually predict better?

Shrinkage is justified by a claim about accuracy, so it should be tested rather than asserted. We hold out one measurement from each assay, fit all three strategies to the rest, and see which predicts the held-out values best.



In [ ]:
set.seed(3)
held <- sapply(1:J, function(j) { idx <- which(codes == j); idx[sample(length(idx), 1)] })
train <- setdiff(seq_along(yobs), held)
nj_tr <- as.numeric(table(factor(codes[train], levels = 1:J)))
ybar_tr <- as.numeric(tapply(yobs[train], factor(codes[train], levels = 1:J), mean))
grand_tr <- mean(yobs[train])
tau_e <- mean(tau_post); sig_e <- mean(sig_post)
lam <- tau_e^2 / (tau_e^2 + sig_e^2 / pmax(nj_tr, 1))
partial <- lam * ybar_tr + (1 - lam) * grand_tr
pool_cmp <- rbindlist(lapply(list(list("complete pooling (grand mean)", rep(grand_tr, J)),
                                  list("no pooling (own assay mean)", ybar_tr),
                                  list("partial pooling (hierarchical)", partial)), function(z) {
  err <- yobs[held] - z[[2]]
  data.table(strategy = z[[1]], RMSE = sqrt(mean(err^2)), MAE = mean(abs(err)),
             `RMSE on assays with n<=5` = sqrt(mean(err[nj <= 5]^2)))
}))
cat(sprintf("held-out measurements: %d (one per assay) | training measurements: %d\n",
            length(held), length(train)))
cat(sprintf("estimated shrinkage weight lambda: min %.2f (n=%d), max %.3f (n=%d)\n",
            min(lam), nj_tr[which.min(lam)], max(lam), nj_tr[which.max(lam)]))
knitr::kable(pool_cmp, digits = 4)



Partial pooling wins on every column: held-out RMSE 0.7333 against 0.7345 for no pooling and 1.1553 for complete pooling, and the same ordering on mean absolute error. The margin over no pooling is small overall because most measurements belong to large assays, where the two strategies coincide — the shrinkage weight λ reaches 0.998 for the largest assay. The third column is where the difference lives: restricted to assays with five or fewer measurements, partial pooling (0.959) beats using the assay's own mean (0.995), and both beat ignoring the labels (1.160). That is the argument in three numbers: with three measurements an assay mean is noisy enough that pulling it towards the grand mean improves the prediction, and the hierarchical model finds the right compromise without being told which regime it is in. This is a cross-validation of a Bayesian model, and it is the kind of evidence that persuades a referee who does not care about priors.

A posterior also answers questions a point estimate cannot, by simple counting over the draws.



In [ ]:
ord <- order(-nj); a_idx <- ord[1]; b_idx <- ord[2]; small_idx <- order(nj)[1]
cat(sprintf("P(assay %s has higher mean potency than assay %s) = %.3f\n",
            assay_ids[a_idx], assay_ids[b_idx], mean(theta_post[, a_idx] > theta_post[, b_idx])))
cat(sprintf("posterior mean difference %.3f log units, 95%% CrI [%.3f, %.3f]\n",
            mean(theta_post[, a_idx] - theta_post[, b_idx]),
            quantile(theta_post[, a_idx] - theta_post[, b_idx], 0.025),
            quantile(theta_post[, a_idx] - theta_post[, b_idx], 0.975)))
cat(sprintf("P(the smallest assay, n=%d, has mean potency above 7) = %.3f  (its raw mean is %.3f)\n",
            nj[small_idx], mean(theta_post[, small_idx] > 7), ybar[small_idx]))
cat(sprintf("P(between-assay SD tau exceeds the within-assay SD sigma) = %.3f\n", mean(tau_post > sig_post)))
set.seed(5)
new_assay <- mu_post + tau_post * rnorm(length(mu_post))
cat(sprintf("posterior predictive for a NEW assay's mean: %.3f, 95%% CrI [%.3f, %.3f]\n",
            mean(new_assay), quantile(new_assay, 0.025), quantile(new_assay, 0.975)))



Each of these is a sentence a chemist can act on, obtained by counting draws rather than by consulting a table: the largest assay (501 compounds) sits 1.038 log units *below* the second largest (305 compounds), with a 95 % credible interval from −1.153 to −0.920 and a posterior probability of 0.000 that it is the more potent of the two — so the two are not interchangeable sources of training labels; the three-compound assay whose raw mean is 6.400 has only a 10.5 % posterior probability of being above pActivity 7; and an assay that has not been run yet would have a mean somewhere in 5.06 to 8.45 — a three-log-unit predictive interval, which is the honest summary of how heterogeneous this literature is. None of these statements has a frequentist counterpart that can be phrased as a probability about the quantity itself<sup class="c"><a href="#r30">[30]</a></sup>.

### 7.2 The three-replica problem, revisited

Units 7 and 8 left the molecular-dynamics comparison unresolved: three replicas per condition, a difference of 0.02 nm in mean C-alpha RMSD, p = 0.69, and an exact permutation test that could not have gone below 0.10. A Bayesian analysis cannot create information that the design did not collect — but it can state the result in a form that does not invite the wrong conclusion.



In [ ]:
md_obs <- fread(D("md_trpcage_observables.csv"))
run_means <- md_obs[, .(m = mean(rmsd_ca_nm)), by = .(condition, replica)]
A <- run_means[condition == "A", m]; B <- run_means[condition == "B", m]
sd_rep <- sqrt((sum((A - mean(A))^2) + sum((B - mean(B))^2)) / (length(A) + length(B) - 2))
normal_posterior <- function(x, prior_mu = 0.20, prior_sd = 0.15, sd_known = sd_rep) {
  prec <- 1 / prior_sd^2 + length(x) / sd_known^2
  c(mean = (prior_mu / prior_sd^2 + sum(x) / sd_known^2) / prec, sd = 1 / sqrt(prec))
}
pa <- normal_posterior(A); pb <- normal_posterior(B)
set.seed(0)
diff_md <- rnorm(200000, pa["mean"], pa["sd"]) - rnorm(200000, pb["mean"], pb["sd"])
xs <- seq(0.02, 0.32, length.out = 400)
dd <- rbind(data.table(x = xs, d = dnorm(xs, pa["mean"], pa["sd"]), what = "300 K mean RMSD"),
            data.table(x = xs, d = dnorm(xs, pb["mean"], pb["sd"]), what = "340 K mean RMSD"))
p <- ggplot(dd, aes(x, d, colour = what)) + geom_line(linewidth = 0.8) +
  geom_point(data = data.table(x = c(A, B), d = rep(c(0.3, 0.6), each = 3),
                               what = rep(c("300 K mean RMSD", "340 K mean RMSD"), each = 3)),
             size = 1.6) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")]), name = NULL) +
  labs(x = "mean C-alpha RMSD (nm)", y = "posterior density",
       title = "Three replicas per condition: the posteriors overlap almost completely")
print(p)
cat(sprintf("replica SD (pooled, 4 df): %.4f nm\n", sd_rep))
cat(sprintf("300 K posterior mean %.4f, 95%% CrI [%.4f, %.4f]\n",
            pa["mean"], pa["mean"] - 1.96 * pa["sd"], pa["mean"] + 1.96 * pa["sd"]))
cat(sprintf("340 K posterior mean %.4f, 95%% CrI [%.4f, %.4f]\n",
            pb["mean"], pb["mean"] - 1.96 * pb["sd"], pb["mean"] + 1.96 * pb["sd"]))
cat(sprintf("difference: posterior mean %+.4f nm, 95%% CrI [%+.4f, %+.4f]\n",
            mean(diff_md), quantile(diff_md, 0.025), quantile(diff_md, 0.975)))
cat(sprintf("P(300 K is MORE mobile than 340 K) = %.3f   <- barely different from a coin flip\n",
            mean(diff_md > 0)))
cat(sprintf("P(|difference| < 0.05 nm) = %.3f  <- the useful statement: any real effect is small\n",
            mean(abs(diff_md) < 0.05)))



<p class="caption"><b>Figure 9.</b> Posterior distributions for the mean C-alpha RMSD at each temperature, built from the three replica means per condition (dots) with a weakly informative Normal(0.20, 0.15) prior and the pooled replica SD of 0.0562 nm. The two posteriors overlap almost entirely. The posterior probability that the 300 K system is the more mobile one is 0.667 — leaning that way, but nowhere near a conclusion — while the probability that the two conditions differ by less than 0.05 nm is 0.693.</p>

Compare the three available reports of the same experiment. The frequentist test says "p = 0.69, no significant difference", which readers routinely misread as "no difference". The exact permutation test says "p = 0.80, and 0.10 was the floor", which is honest but conveys nothing about size. The Bayesian analysis says **"the 300 K minus 340 K difference is +0.019 nm with a 95 % credible interval from −0.069 to +0.107, and the probability that it is smaller than 0.05 nm in magnitude is 0.69"** — an explicit statement that the experiment has bounded the effect loosely rather than refuted it. That is the sentence to write in a thesis, and it is the clearest argument in this course for learning the Bayesian vocabulary even if the analysis itself stays frequentist<sup class="c"><a href="#r31">[31]</a></sup>.

For contrast, the next chunk runs the identical machinery on a comparison that *is* well powered — the two ROCK1 scaffold series of Unit 8 — so that the difference between "no evidence of an effect" and "evidence of no large effect" is visible side by side.



In [ ]:
rock <- fread(D("rock1_chembl_curated.csv"))
ex_c <- rock[censored == FALSE]
cnt <- sort(table(ex_c$scaffold_smiles), decreasing = TRUE)
s1 <- ex_c[scaffold_smiles == names(cnt)[1], pActivity]
s2 <- ex_c[scaffold_smiles == names(cnt)[2], pActivity]
sd_pool <- sqrt(((length(s1) - 1) * var(s1) + (length(s2) - 1) * var(s2)) / (length(s1) + length(s2) - 2))
p1 <- normal_posterior(s1, 6.5, 2.0, sd_pool); p2 <- normal_posterior(s2, 6.5, 2.0, sd_pool)
set.seed(2)
d12 <- rnorm(200000, p1["mean"], p1["sd"]) - rnorm(200000, p2["mean"], p2["sd"])
cat(sprintf("series 1: n = %3d, mean %.3f | series 2: n = %2d, mean %.3f | pooled SD %.3f\n",
            length(s1), mean(s1), length(s2), mean(s2), sd_pool))
cat(sprintf("posterior difference %+.3f log units, 95%% CrI [%+.3f, %+.3f]\n",
            mean(d12), quantile(d12, 0.025), quantile(d12, 0.975)))
cat(sprintf("P(series 1 more potent than series 2)      = %.4f\n", mean(d12 > 0)))
cat(sprintf("P(the difference exceeds 1 log unit)       = %.4f\n", mean(d12 > 1)))
cat(sprintf("P(the difference is smaller than 0.3 log)  = %.4f\n", mean(abs(d12) < 0.3)))
cat(sprintf("\nfor comparison, the MD replica question: P(difference > 0) = %.3f, P(|difference| < 0.05 nm) = %.3f\n",
            mean(diff_md > 0), mean(abs(diff_md) < 0.05)))



With 207 and 59 compounds the posterior for the difference is tight and far from zero: series 1 is more potent with a probability indistinguishable from 1, the difference is +1.325 log units (95 % CrI 1.078 to 1.573), and the probability that it exceeds one log unit is 0.995. With three replicas per condition the posterior for the molecular-dynamics difference straddles zero and the probability of a positive difference is 0.667. Both analyses used the same three lines of code; what differs is the information in the data, which is precisely what a posterior is designed to display.

<div class="exercise"><h4>Exercise 2</h4>
<p>Raise the lower bound on assay size from 3 to 10 measurements (<code>ok[n_in_assay &gt;= 10]</code>) and re-run the sampler. Does the between-assay standard deviation tau go up or down, and does the average shrinkage increase or decrease? Then set the prior on tau to HalfNormal(0.1) — a strong belief that assays agree — by changing the <code>-0.5 * t^2</code> term to <code>-0.5 * (t / 0.1)^2</code>, and describe what happens to the posterior means of the small assays.</p></div>

## 8. Predictive checks

A posterior can be precise and wrong, because precision is a statement about the parameter *within the model* and says nothing about whether the model could have produced the data. The check is predictive: simulate datasets from the model and compare them with the real one.

- A **prior predictive check** draws parameters from the prior and simulates data, before looking at the observations. It asks whether the priors imply anything absurd — pActivity values of 40, negative hit rates, potencies spanning twenty log units.
- A **posterior predictive check** draws parameters from the posterior and simulates data, then compares a chosen summary of the simulations with the same summary of the real data<sup class="c"><a href="#r20">[20]</a></sup>.



In [ ]:
kurtosis <- function(x) mean((x - mean(x))^4) / mean((x - mean(x))^2)^2 - 3
skewness <- function(x) mean((x - mean(x))^3) / mean((x - mean(x))^2)^1.5
set.seed(1)
prior_mu <- rnorm(400, 6.5, 2.0); prior_tau <- abs(rnorm(400, 0, 1)); prior_sig <- abs(rnorm(400, 0, 1))
prior_sim <- t(sapply(1:400, function(i)
  rnorm(50, prior_mu[i] + prior_tau[i] * rnorm(50), prior_sig[i])))
idx <- sample(nrow(post_h), 400)
post_sim <- t(sapply(idx, function(i) rnorm(length(yobs), theta_post[i, codes], sig_post[i])))
op <- par(mfrow = c(1, 3), mar = c(4, 4, 2.6, 1), cex = 0.72)
plot(NA, xlim = c(0, 14), ylim = c(0, 1), xlab = "pActivity", ylab = "cumulative fraction",
     main = "prior predictive (grey) vs data")
for (i in 1:120) lines(sort(prior_sim[i, ]), seq(0, 1, length.out = 50), col = "#bbbbbb66", lwd = 0.5)
lines(sort(yobs), seq(0, 1, length.out = length(yobs)), col = PAL["orange"], lwd = 2)
plot(NA, xlim = c(2, 12), ylim = c(0, 1), xlab = "pActivity", ylab = "",
     main = "posterior predictive (blue) vs data")
for (i in 1:120) lines(sort(post_sim[i, ]), seq(0, 1, length.out = ncol(post_sim)),
                       col = adjustcolor(PAL["blue"], 0.25), lwd = 0.4)
lines(sort(yobs), seq(0, 1, length.out = length(yobs)), col = PAL["orange"], lwd = 2)
ks <- apply(post_sim, 1, kurtosis)
hist(ks, breaks = 40, col = "grey80", border = NA, xlim = range(c(ks, kurtosis(yobs))),
     xlab = "excess kurtosis of a simulated dataset", main = "a test statistic the model misses")
abline(v = kurtosis(yobs), col = PAL["orange"], lwd = 2)
par(op)
cat(sprintf("prior predictive: 2.5-97.5 %% range of simulated pActivity = [%.1f, %.1f] (data: [%.1f, %.1f])\n",
            quantile(prior_sim, 0.025), quantile(prior_sim, 0.975), min(yobs), max(yobs)))
cat(sprintf("posterior predictive mean %.3f vs observed %.3f\n", mean(post_sim), mean(yobs)))
cat(sprintf("posterior predictive SD   %.3f vs observed %.3f\n", sd(as.vector(post_sim)), sd(yobs)))
cat(sprintf("observed excess kurtosis %.3f vs simulated %.3f [%.3f, %.3f]; posterior predictive p = %.4f\n",
            kurtosis(yobs), mean(ks), quantile(ks, 0.025), quantile(ks, 0.975), mean(ks <= kurtosis(yobs))))
cat(sprintf("observed skewness %.3f (the model reproduces this one: p = %.3f)\n",
            skewness(yobs), mean(apply(post_sim, 1, skewness) >= skewness(yobs))))
cat(sprintf("observed minimum %.2f vs simulated minima %.2f [%.2f, %.2f]\n",
            min(yobs), mean(apply(post_sim, 1, min)),
            quantile(apply(post_sim, 1, min), 0.025), quantile(apply(post_sim, 1, min), 0.975)))



<p class="caption"><b>Figure 10.</b> Predictive checks (orange: the observed data). Left: datasets simulated from the prior alone (grey) are far too broad — the priors permit pActivity values from below 2 to above 11 — which is acceptable for a weakly informative prior but would be a problem if they excluded the observed range. Centre: datasets simulated from the posterior (blue) track the observed distribution closely through the bulk, reproducing the mean to three decimals (6.545) and the standard deviation to three (1.125). Right: the model nevertheless fails one specific test — the excess kurtosis of the real data, −0.474, lies below every one of the 400 simulated values (simulated range −0.210 to +0.097, posterior predictive p = 0.000). The real potencies are <em>lighter</em>-tailed than a normal likelihood implies, because every assay is bounded by the concentration range it was run over; the simulated datasets contain compounds weaker (minimum 2.74 against the observed 3.57) and stronger than any real measurement.</p>

The right-hand panel is the point of the exercise. A posterior predictive check is only as good as the summary statistic chosen, and the useful ones are those the model was *not* fitted to match. Means and standard deviations will almost always agree, because the model has parameters for them — and indeed they match to three decimals here. Skewness happens to pass as well (p = 0.88). Kurtosis, extreme quantiles, the number of zeros, or the autocorrelation of a time series are the statistics that reveal misfit, and one of them does. The honest conclusion for our model: it captures the centre, spread and asymmetry of the assay structure and gets the tails wrong, so posterior intervals for central quantities are trustworthy and predictions about extreme compounds are not.

## 9. The same model with lme4 — and what a probabilistic programming language would add

Writing samplers by hand is the right way to learn and the wrong way to work. The Python edition of this unit notes that its models are hand-written because the compiler behind PyMC<sup class="c"><a href="#r21">[21]</a></sup><sup class="c"><a href="#r22">[22]</a></sup> does not run on the build machine, and gives the PyMC code as text. The R edition has a better option for *this particular model*: `lme4::lmer`<sup class="c"><a href="#r2">[2]</a></sup> fits the same hierarchical normal model by restricted maximum likelihood in one line, and comparing the two is the strongest available check on fifty lines of hand-written Metropolis-within-Gibbs.



In [ ]:
dt_assay <- data.table(y = yobs, assay = factor(codes))
mm <- lmer(y ~ 1 + (1 | assay), data = dt_assay, REML = TRUE)
vc <- as.data.frame(VarCorr(mm))
blups <- coef(mm)$assay[, 1]
cat(sprintf("lmer   : intercept %.3f | between-assay SD %.3f | residual SD %.3f\n",
            fixef(mm)[1], vc$sdcor[1], vc$sdcor[2]))
cat(sprintf("Bayes  : mu        %.3f | tau            %.3f | sigma       %.3f\n",
            mean(mu_post), mean(tau_post), mean(sig_post)))
cat(sprintf("correlation between lmer BLUPs and Bayesian posterior means: %.4f\n", cor(blups, theta_hat)))
cat(sprintf("mean |difference| between the two sets of 108 estimates: %.4f log units\n",
            mean(abs(blups - theta_hat))))
cat(sprintf("largest |difference|: %.4f log units (assay %s, n = %d)\n",
            max(abs(blups - theta_hat)), assay_ids[which.max(abs(blups - theta_hat))],
            nj[which.max(abs(blups - theta_hat))]))
p <- ggplot(data.table(blup = blups, bayes = theta_hat, n = nj, raw = ybar),
            aes(blup, bayes, size = n)) +
  geom_abline(linetype = 2) +
  geom_point(colour = PAL["blue"], alpha = 0.6) +
  scale_size_continuous(range = c(0.6, 3), trans = "log10", name = "measurements") +
  labs(x = "lme4 conditional mode (BLUP) of the assay effect",
       y = "Bayesian posterior mean",
       title = "Two routes to partial pooling, 108 assays, one answer")
print(p)



<p class="caption"><b>Figure 11.</b> The 108 assay estimates from the hand-written sampler against those from <code>lmer</code>, with point size proportional to assay size and the identity line dashed. The two agree to a correlation of 1.0000, a mean absolute difference of 0.0016 log units and a worst case of about 0.01 — far below the Monte-Carlo noise of the sampler, and three orders of magnitude below the between-assay standard deviation. The variance components agree too: between-assay 0.836 (lmer) against 0.843 (Bayes), within-assay 0.828 against 0.828. The small systematic gap in the between-assay term is expected: REML returns a point estimate of τ, while the Bayesian posterior mean averages over the uncertainty in τ, which is asymmetric and therefore slightly larger.</p>

This agreement is worth dwelling on, because it tells a student which tool to reach for. For a **balanced-ish hierarchical normal model with no further structure**, `lmer` gives the same partial pooling in one line and in a fraction of a second, with the empirical-Bayes conditional modes that genomics calls shrinkage estimates. What it does not give is a distribution: there is no posterior for τ, no probability that one assay exceeds another, no predictive interval for an unrun assay, and the uncertainty in the variance components is not propagated into the group estimates. Those are the four things §7 produced by counting draws, and they are the reason to pay the extra cost when the question needs them.

For models beyond this one — non-normal likelihoods, censored potencies, several nested levels, a hierarchical regression with varying slopes — the right tool is a probabilistic programming language rather than either of these. In R that is **brms**<sup class="c"><a href="#r32">[32]</a></sup> or **rstanarm**, both front-ends to Stan<sup class="c"><a href="#r23">[23]</a></sup>, in which this unit's model is one line of formula syntax:



In [ ]:
## Not run here: brms/rstanarm require a Stan toolchain, which this build environment lacks.
## In a Colab R runtime, install the brms package once, then:
library(brms)
fit <- brm(y ~ 1 + (1 | assay), data = dt_assay,
           prior = c(prior(normal(6.5, 2), class = "Intercept"),
                     prior(normal(0, 1), class = "sd"),        ## HalfNormal(1) on tau
                     prior(normal(0, 1), class = "sigma")),    ## HalfNormal(1) on sigma
           chains = 4, iter = 2000, seed = 0)
summary(fit)                 ## posterior means, 95 % CrI, R-hat and ESS for every parameter
pp_check(fit, ndraws = 100)  ## the posterior predictive check of section 8, in one call



<div class="box warn"><h4>Substitution in the R edition — no probabilistic programming language</h4>
<p>The chunk above is shown but <strong>not evaluated</strong> (<code>eval = FALSE</code>): <code>brms</code> and <code>rstanarm</code> need a working C++ toolchain and a compiled Stan backend, which the environment this unit is knitted in does not provide. Everything in this unit therefore runs on base R, and the hand-written sampler of §7 is validated against <code>lme4</code> in Figure 11 rather than against Stan. This is in any case the better teaching path — <strong>if you cannot write a Metropolis sampler, you cannot diagnose one</strong> — and the code above runs unchanged in Colab once the <code>brms</code> package has been installed there. Two details of it are worth copying: <code>brms</code> uses the <strong>non-centred parameterisation</strong> internally (sampling standard normals and transforming, rather than sampling the group effects directly), which avoids the funnel-shaped geometry that makes hierarchical models hard to sample, and <code>summary()</code> reports R-hat and effective sample size automatically, which is the main practical reason to use it.</p></div>

Priors for variance parameters deserve one specific remark, because the default choice in older textbooks is bad: an inverse-gamma prior with tiny parameters is not "uninformative" for a between-group standard deviation and can dominate the posterior when the number of groups is small. A half-normal or half-Cauchy on the standard deviation itself, as used in §7 and in the `brms` block above, is the modern default<sup class="c"><a href="#r24">[24]</a></sup><sup class="c"><a href="#r25">[25]</a></sup>.

## 10. Credible intervals and confidence intervals

The two are different objects that often look the same.

- A **95 % credible interval** contains the parameter with probability 0.95 *given the data and the model*. It is a statement about the parameter.
- A **95 % confidence interval** is produced by a procedure that, over hypothetical repetitions of the experiment, contains the true value 95 % of the time. It is a statement about the procedure, and the interval in front of you either contains the parameter or does not<sup class="c"><a href="#r26">[26]</a></sup>.

With plenty of data and a weak prior the two nearly coincide, which is why the misinterpretation of confidence intervals is so persistent<sup class="c"><a href="#r27">[27]</a></sup>. They separate when data are scarce or the parameter is near a boundary.



In [ ]:
cmp_int <- rbindlist(lapply(list(c(40, k_active), c(400, k400), c(12, 0)), function(z) {
  n_ <- z[1]; k_ <- z[2]; ph <- k_ / n_
  wald <- ph + c(-1.96, 1.96) * sqrt(ph * (1 - ph) / n_)
  wilson <- as.numeric(prop.test(k_, n_, correct = FALSE)$conf.int)
  data.table(n = n_, k = k_, MLE = ph,
             Wald = sprintf("[%.3f, %.3f]", wald[1], wald[2]),
             Wilson = sprintf("[%.3f, %.3f]", wilson[1], wilson[2]),
             Bayes_mean = (a0 + k_) / (a0 + b0 + n_),
             credible = sprintf("[%.3f, %.3f]", qbeta(0.025, a0 + k_, b0 + n_ - k_),
                                qbeta(0.975, a0 + k_, b0 + n_ - k_)),
             w_lo = wilson[1], w_hi = wilson[2],
             c_lo = qbeta(0.025, a0 + k_, b0 + n_ - k_), c_hi = qbeta(0.975, a0 + k_, b0 + n_ - k_))
}))
plot_dt <- rbind(cmp_int[, .(lab = sprintf("n=%d, k=%d", n, k), est = MLE, lo = w_lo, hi = w_hi,
                             kind = "Wilson confidence interval")],
                 cmp_int[, .(lab = sprintf("n=%d, k=%d", n, k), est = Bayes_mean, lo = c_lo, hi = c_hi,
                             kind = "Bayesian credible interval")])
p <- ggplot(plot_dt, aes(est, lab, colour = kind)) +
  geom_errorbarh(aes(xmin = lo, xmax = hi), height = 0.18, position = position_dodge(width = 0.4)) +
  geom_point(size = 1.8, position = position_dodge(width = 0.4)) +
  scale_colour_manual(values = unname(PAL[c("orange", "blue")]), name = NULL) +
  labs(x = "hit rate", y = NULL, title = "The two intervals agree until the data run out")
print(p)
knitr::kable(cmp_int[, .(n, k, MLE, Wald, Wilson, Bayes_mean, credible)], digits = 3)



<p class="caption"><b>Figure 12.</b> Wilson confidence intervals (orange) and Bayesian credible intervals under the Beta(2,18) prior (blue) for three datasets. At n = 400 they are nearly identical ([0.068, 0.125] against [0.067, 0.122]). At n = 40 they differ by a few hundredths and tell the same story. At n = 12 with zero actives they separate decisively: the maximum-likelihood estimate is exactly 0, the Wald interval collapses to the degenerate [0.000, 0.000] — a statement no one believes — the Wilson interval runs to 0.242, and the credible interval is [0.008, 0.167] with a posterior mean of 0.062, which is what a chemist would actually say after twelve failures.</p>

The boundary case in the table is the practical argument for Bayesian intervals in small-sample work. "Zero of twelve" is extremely common in early screening, in toxicity studies and in rare-event clinical data, and the classical machinery handles it badly without corrections — in R, `prop.test` gives the Wilson interval and `binom.test` the Clopper–Pearson one, both of which behave sensibly, while the textbook Wald formula does not. A weakly informative prior solves it by construction.

<div class="box key"><h4>Which to report</h4>
<p>Report the one whose definition matches the sentence you want to write. If the sentence is "there is a 95 % probability that the hit rate is between 6 % and 23 %", you need a credible interval, and you must state the prior. If the sentence is "a procedure with 95 % long-run coverage gives [7 %, 29 %]", a confidence interval suffices. What is never acceptable is to compute one and describe it as the other<sup class="c"><a href="#r27">[27]</a></sup>.</p></div>

## 11. Model comparison: Bayes factors and information criteria, briefly

A **Bayes factor** is the ratio of the marginal likelihoods of two models, \( BF_{12}=P(D\mid M_1)/P(D\mid M_2) \): how much the data shift the relative odds of the two models<sup class="c"><a href="#r28">[28]</a></sup>. It is attractive because it can support the null hypothesis rather than merely failing to reject it<sup class="c"><a href="#r29">[29]</a></sup>, and dangerous because the marginal likelihood depends on the priors even when the posterior does not — a prior ten times too wide divides the Bayes factor by roughly ten. Computing it requires integrating over the whole prior, which is hard in more than a few dimensions.

Information criteria (AIC and BIC, <a href="../U9_Linear_Models/index.html">Unit 9</a>) approximate related quantities much more cheaply, and the modern Bayesian default for predictive comparison is cross-validated predictive density rather than either. For this course the practical rule is: use posterior distributions to answer questions about parameters, use cross-validation (Unit 12) to compare predictive models, and reach for Bayes factors only with explicit, justified priors and a sensitivity analysis.



In [ ]:
marginal_likelihood <- function(a, b, n, k, grid_n = 20000) {
  t <- seq(1e-6, 1 - 1e-6, length.out = grid_n)
  sum(dbeta(t, a, b) * dbinom(k, n, t)) * diff(t)[1]
}
m_point <- dbinom(k_active, n_tested, 0.10)        ## M2: the hit rate is exactly 10 %
bf <- rbindlist(lapply(list(c(2, 18), c(1, 9), c(20, 180), c(1, 1)), function(pp) {
  data.table(prior = sprintf("Beta(%.1f, %.1f)", pp[1], pp[2]),
             prior_sd = sqrt(pp[1] * pp[2] / (sum(pp)^2 * (sum(pp) + 1))),
             BF_flexible_vs_point = marginal_likelihood(pp[1], pp[2], n_tested, k_active) / m_point,
             posterior_mean = (pp[1] + k_active) / (sum(pp) + n_tested))
}))
cat("the data are identical in every row; only the prior width changed.\n")
knitr::kable(bf, digits = 4)



The table makes the warning concrete: the four priors give posterior means spanning 0.058 (0.108 to 0.167), yet their Bayes factors span a factor of four (0.228 to 0.964), because a wider prior spreads its marginal likelihood over possibilities the data rule out. Note also that every Bayes factor here is below 1, i.e. the data mildly favour the point hypothesis "the rate is exactly 10 %" over the flexible model — unsurprising with 6 active of 40, but a conclusion whose strength depends on a prior the point model does not have. Posterior summaries are robust to the prior here; the Bayes factor is not. That asymmetry is the reason this course teaches estimation rather than Bayesian hypothesis testing.

<div class="box bridge"><h4>Bridge</h4>
<p>This unit is the Bayesian counterpart of the block that preceded it: <a href="../U7_Estimation/index.html">Unit 7</a>'s confidence intervals become credible intervals in §10, <a href="../U8_Hypothesis_Testing/index.html">Unit 8</a>'s p-values become posterior probabilities in §2 and §11, and <a href="../U9_Linear_Models/index.html">Unit 9</a>'s blocking factor and its <code>lmer</code> fit become the hierarchical model of §7, cross-checked against <code>lmer</code> in §9. The Python counterpart of this unit is <a href="../../Python/U10_Bayesian/index.html">here</a> and the R edition is indexed at <a href="../index.html">the course root</a>. Forward, the base-rate arithmetic of §2 is the foundation of <a href="../U19_Virtual_Screening/index.html">Unit 19</a>'s enrichment metrics, the partial pooling of §7 is how <a href="../U20_MD_Analytics/index.html">Unit 20</a> should combine replica averages, <a href="../U17_Uncertainty_AD/index.html">Unit 17</a> builds distribution-free prediction intervals as the non-Bayesian route to the same goal, and <a href="../U21_Omics/index.html">Unit 21</a> uses the empirical-Bayes shrinkage of gene-wise variances that §7 motivates. In the sister courses, the bioinformatics unit on <a href="../../../BGM324/03_Molecular_Docking/index.html">molecular docking</a> produces the ranked lists whose hit probability §2 computes, the cheminformatics unit on <a href="../../../Cheminformatics/U6_Databases/index.html">chemical databases</a> explains where the 108 assays of §7 come from, and the R edition of the AI course, indexed <a href="../../../BGM411/R/index.html">here</a>, covers naive Bayes and probabilistic classifiers.</p></div>

## 12. Summary

You should be able to explain, without notes:

- Bayes' theorem in words, and why the posterior requires a prior as well as a likelihood;
- the base-rate fallacy, and why a screen with an enrichment factor of 8 on a 0.1 %-active library still delivers 99 inactive compounds in every 100 ordered;
- conjugate updating for the beta–binomial — prior Beta(2,18) plus 6 of 40 gives Beta(8,52), mean 0.1333 — and that the posterior mean is a weighted compromise between prior and data;
- how to compute a posterior by grid approximation, and the dimensional limit that makes it fail;
- how a Metropolis sampler works, why the normalising constant cancels, and the three diagnostics (acceptance rate, R-hat, effective sample size) that separated a healthy chain from one with 17 effective draws;
- what partial pooling does, why small assays shrank by 0.151 log units and large ones by 0.008, and which of the group's data structures are hierarchical;
- that `lmer` reproduces the same 108 partially pooled estimates to a correlation of 1.0000, and what it does *not* give;
- what prior and posterior predictive checks test, and why the useful test statistic (kurtosis, p = 0.000) is one the model was not fitted to match;
- the difference between a credible and a confidence interval, and the zero-of-twelve case where Wald gives [0, 0] and the credible interval gives [0.008, 0.167];
- why posterior summaries are usually robust to the prior while Bayes factors moved by a factor of four over the same four priors;
- how to report a Bayesian analysis: model, priors, sampler diagnostics, posterior summaries, prior sensitivity, predictive check.

## 13. Self-assessment

<details><summary>1. A test with 99 % sensitivity and 95 % specificity is positive. Why might you still be unlikely to be ill?</summary>Because the posterior depends on the prior prevalence. With a disease frequency of 1 in 1,000, the 5 % false-positive rate applied to the 99.9 % healthy majority produces about fifty times more false positives than true positives, giving P(ill | positive) = 99/5,094 ≈ 1.9 %.</details>

<details><summary>2. Your virtual screen has EF(1 %) = 20 on a library whose prevalence of actives is 0.05 %. You order 50 compounds. How many actives do you expect?</summary>P(active | selected) ≈ 20 × 0.0005 = 0.01, so about 0.5 actives among 50 compounds. This is the arithmetic behind "order a hundred, not five", and the reason an experimental collaborator asks how many compounds the screen is confident about.</details>

<details><summary>3. Prior Beta(2,18), data 6 active of 40. What is the posterior and its mean?</summary>Beta(2+6, 18+34) = Beta(8, 52), with mean 8/60 = 0.1333 — between the prior mean of 0.10 and the data estimate of 0.15, closer to the data because 40 observations outweigh 20 observations' worth of prior. Its 95 % credible interval is [0.060, 0.229].</details>

<details><summary>4. Why does the normalising constant not matter in the Metropolis algorithm?</summary>Because the accept/reject decision depends only on the ratio of posterior densities at two points, and the unknown constant P(D) appears in both numerator and denominator. This is what makes MCMC possible for models whose evidence integral cannot be computed — and it is why the sampler in §6 needs only <code>dbeta(..., log = TRUE) + dbinom(..., log = TRUE)</code>.</details>

<details><summary>5. Your chain has an acceptance rate of 0.97. Is that good?</summary>No — it means the proposal steps are far too small, so the chain crawls and consecutive draws are nearly identical. In §6 that chain had an effective sample size of 17 per chain against 687 for the well-tuned one, and R-hat 1.060, even though its posterior mean was correct to three decimals. For a random-walk Metropolis sampler, aim for roughly 0.2–0.5 and check R-hat and ESS rather than the trace alone.</details>

<details><summary>6. What is R-hat and what value is acceptable?</summary>The ratio of an estimate of the total posterior variance (pooling chains) to the within-chain variance; if chains started from different places have converged to the same distribution it approaches 1. Values above about 1.01 indicate that the chains have not mixed, and the posterior summaries should not be used.</details>

<details><summary>7. An assay with three compounds has a mean pActivity of 7.97; the hierarchical model reports 7.67. Has the model thrown away information?</summary>No — it has combined the three measurements with the information that assays in this dataset are distributed around 6.75 with a between-assay SD of 0.84. A mean of 7.97 from three measurements is weak evidence of an exceptional assay and strong evidence of a small sample. The shrunken estimate is the better prediction for the next compound in that assay, as the held-out test of §7.1 confirms (RMSE 0.959 against 0.995 for assays with n ≤ 5).</details>

<details><summary>8. When does shrinkage not happen?</summary>When the group has enough data for its own mean to be precise (λ → 1 as n grows; it reached 0.998 for the 501-compound assay), or when the between-group variance tau is large relative to the within-group variance, meaning the groups really are different and pooling would be wrong. The model estimates tau from the data rather than assuming it, which is the point.</details>

<details><summary>9. Your posterior predictive check shows the model reproduces the mean and SD but not the kurtosis. What now?</summary>Report it, and decide whether it matters for the question. The model has parameters for the mean and the variance, so matching those is not evidence; the failure on kurtosis (observed −0.474, simulated −0.210 to +0.097) says the normal likelihood is wrong in the tails. If the conclusion concerns central quantities it stands; if it concerns extreme compounds, use a heavier-tailed likelihood such as Student's t.</details>

<details><summary>10. Twelve compounds tested, none active. What do you report?</summary>Not "the hit rate is 0". Report a credible interval under a stated prior — [0.008, 0.167] under Beta(2,18) — or a Wilson confidence interval ([0.000, 0.242] from <code>prop.test</code>), and state explicitly that zero of twelve is compatible with a hit rate of 10 %. The Wald interval of [0, 0] is an artefact of the formula.</details>

<details><summary>11. Why are posterior means robust to the prior here while Bayes factors are not?</summary>Because the posterior renormalises over the parameter space and the likelihood dominates once there are enough data, whereas the marginal likelihood integrates the likelihood against the prior, so a wider prior dilutes it. In §11 four priors moved the posterior mean by 0.058 and the Bayes factor by a factor of four.</details>

<details><summary>12. A referee objects that your prior is subjective. What is the answer?</summary>That every modelling choice is, and that this one is written down. Show the prior predictive check, show the sensitivity table across several defensible priors (§5), and show that the conclusion is unchanged — or, if it changes, say so, because then the data are not sufficient to support it without the prior. For the hierarchical model you can add the strongest check of all: <code>lmer</code>, which uses no prior at all, gives the same 108 estimates.</details>

## Glossary

<dl class="gloss">
<dt>Base-rate fallacy</dt><dd>Ignoring the prior prevalence when interpreting a positive test or a predicted hit.</dd>
<dt>Bayes factor</dt><dd>Ratio of two models' marginal likelihoods; sensitive to the priors even when posteriors are not.</dd>
<dt>BLUP / conditional mode</dt><dd>The mixed-model point estimate of a group effect; the frequentist analogue of a posterior mean under partial pooling.</dd>
<dt>Conjugate prior</dt><dd>A prior whose family is preserved by the update, making the posterior available in closed form (beta–binomial, normal–normal).</dd>
<dt>Credible interval</dt><dd>An interval containing the parameter with a stated posterior probability, given data, model and prior.</dd>
<dt>Effective sample size</dt><dd>The number of independent draws an autocorrelated chain is worth.</dd>
<dt>Enrichment factor</dt><dd>Hit rate among the selected fraction divided by the library prevalence; capped at 1/prevalence.</dd>
<dt>Grid approximation</dt><dd>Evaluating prior × likelihood on a grid and normalising; exact in one or two dimensions, hopeless beyond three.</dd>
<dt>Hierarchical model</dt><dd>A model in which group parameters are themselves draws from a common distribution, producing partial pooling.</dd>
<dt>Likelihood</dt><dd>The probability of the observed data as a function of the parameters.</dd>
<dt>Marginal likelihood (evidence)</dt><dd>The probability of the data averaged over the prior; the denominator of Bayes' theorem.</dd>
<dt>MCMC</dt><dd>Markov chain Monte Carlo: a random walk whose stationary distribution is the posterior.</dd>
<dt>Metropolis algorithm</dt><dd>Propose, compute the density ratio, accept with probability min(1, r); the ancestor of all MCMC.</dd>
<dt>Metropolis-within-Gibbs</dt><dd>Conjugate (Gibbs) draws for the parameters that allow them, Metropolis steps for the rest; the sampler of §7.</dd>
<dt>Non-centred parameterisation</dt><dd>Sampling a standard normal and transforming, to avoid the funnel geometry of hierarchical posteriors.</dd>
<dt>Partial pooling</dt><dd>Estimating a group parameter as a compromise between its own data and the overall distribution.</dd>
<dt>Posterior</dt><dd>The distribution of the parameters given the data; the output of a Bayesian analysis.</dd>
<dt>Posterior predictive check</dt><dd>Simulating datasets from the posterior and comparing a summary with the observed data.</dd>
<dt>Prior</dt><dd>The distribution of the parameters before seeing the data; an assumption to be stated and varied.</dd>
<dt>R-hat</dt><dd>Convergence diagnostic comparing between-chain and within-chain variance; should be below about 1.01.</dd>
<dt>Shrinkage</dt><dd>Movement of a group's estimate towards the overall mean, by an amount decreasing with the group's sample size.</dd>
<dt>Weakly informative prior</dt><dd>A prior that excludes absurd values without determining the answer; the practical default.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r2">Bates D, Mächler M, Bolker B, Walker S. Fitting linear mixed-effects models using lme4. <em>J Stat Softw</em>. 2015;67(1):1-48. <a href="https://doi.org/10.18637/jss.v067.i01">doi:10.18637/jss.v067.i01</a></li>
<li id="r3">Ioannidis JPA. Why most published research findings are false. <em>PLoS Med</em>. 2005;2(8):e124. <a href="https://doi.org/10.1371/journal.pmed.0020124">doi:10.1371/journal.pmed.0020124</a></li>
<li id="r4">Colquhoun D. The reproducibility of research and the misinterpretation of p-values. <em>R Soc Open Sci</em>. 2017;4:171085. <a href="https://doi.org/10.1098/rsos.171085">doi:10.1098/rsos.171085</a></li>
<li id="r5">Truchon JF, Bayly CI. Evaluating virtual screening methods: good and bad metrics for the "early recognition" problem. <em>J Chem Inf Model</em>. 2007;47(2):488-508. <a href="https://doi.org/10.1021/ci600426e">doi:10.1021/ci600426e</a></li>
<li id="r6">Mysinger MM, Carchia M, Irwin JJ, Shoichet BK. Directory of Useful Decoys, Enhanced (DUD-E): better ligands and decoys for better benchmarking. <em>J Med Chem</em>. 2012;55(14):6582-6594. <a href="https://doi.org/10.1021/jm300687e">doi:10.1021/jm300687e</a></li>
<li id="r7">Gigerenzer G, Hoffrage U. How to improve Bayesian reasoning without instruction: frequency formats. <em>Psychol Rev</em>. 1995;102(4):684-704. <a href="https://doi.org/10.1037/0033-295X.102.4.684">doi:10.1037/0033-295X.102.4.684</a></li>
<li id="r8">Casscells W, Schoenberger A, Graboys TB. Interpretation by physicians of clinical laboratory results. <em>N Engl J Med</em>. 1978;299(18):999-1001. <a href="https://doi.org/10.1056/NEJM197811022991808">doi:10.1056/NEJM197811022991808</a></li>
<li id="r9">Metropolis N, Rosenbluth AW, Rosenbluth MN, Teller AH, Teller E. Equation of state calculations by fast computing machines. <em>J Chem Phys</em>. 1953;21(6):1087-1092. <a href="https://doi.org/10.1063/1.1699114">doi:10.1063/1.1699114</a></li>
<li id="r10">Hastings WK. Monte Carlo sampling methods using Markov chains and their applications. <em>Biometrika</em>. 1970;57(1):97-109. <a href="https://doi.org/10.1093/biomet/57.1.97">doi:10.1093/biomet/57.1.97</a></li>
<li id="r11">Geman S, Geman D. Stochastic relaxation, Gibbs distributions, and the Bayesian restoration of images. <em>IEEE Trans Pattern Anal Mach Intell</em>. 1984;6(6):721-741. <a href="https://doi.org/10.1109/TPAMI.1984.4767596">doi:10.1109/TPAMI.1984.4767596</a></li>
<li id="r12">Neal RM. MCMC using Hamiltonian dynamics. In: <em>Handbook of Markov Chain Monte Carlo</em>. Chapman &amp; Hall/CRC; 2011:113-162. <a href="https://doi.org/10.1201/b10905-6">doi:10.1201/b10905-6</a></li>
<li id="r13">Gelman A, Rubin DB. Inference from iterative simulation using multiple sequences. <em>Stat Sci</em>. 1992;7(4):457-472. <a href="https://doi.org/10.1214/ss/1177011136">doi:10.1214/ss/1177011136</a></li>
<li id="r14">Vehtari A, Gelman A, Simpson D, Carpenter B, Bürkner PC. Rank-normalization, folding, and localization: an improved R-hat for assessing convergence of MCMC. <em>Bayesian Anal</em>. 2021;16(2):667-718. <a href="https://doi.org/10.1214/20-BA1221">doi:10.1214/20-BA1221</a></li>
<li id="r15">Efron B, Morris C. Stein's paradox in statistics. <em>Sci Am</em>. 1977;236(5):119-127. <a href="https://doi.org/10.1038/scientificamerican0577-119">doi:10.1038/scientificamerican0577-119</a></li>
<li id="r16">Smyth GK. Linear models and empirical Bayes methods for assessing differential expression in microarray experiments. <em>Stat Appl Genet Mol Biol</em>. 2004;3:Article 3. <a href="https://doi.org/10.2202/1544-6115.1027">doi:10.2202/1544-6115.1027</a></li>
<li id="r17">Storey JD, Tibshirani R. Statistical significance for genomewide studies. <em>Proc Natl Acad Sci USA</em>. 2003;100(16):9440-9445. <a href="https://doi.org/10.1073/pnas.1530509100">doi:10.1073/pnas.1530509100</a></li>
<li id="r18">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data - a statistical analysis. <em>PLoS ONE</em>. 2013;8(4):e61007. <a href="https://doi.org/10.1371/journal.pone.0061007">doi:10.1371/journal.pone.0061007</a></li>
<li id="r19">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r20">Gabry J, Simpson D, Vehtari A, Betancourt M, Gelman A. Visualization in Bayesian workflow. <em>J R Stat Soc A</em>. 2019;182(2):389-402. <a href="https://doi.org/10.1111/rssa.12378">doi:10.1111/rssa.12378</a></li>
<li id="r21">Abril-Pla O, Andreani V, Carroll C, et al. PyMC: a modern, and comprehensive probabilistic programming framework in Python. <em>PeerJ Comput Sci</em>. 2023;9:e1516. <a href="https://doi.org/10.7717/peerj-cs.1516">doi:10.7717/peerj-cs.1516</a></li>
<li id="r22">Salvatier J, Wiecki TV, Fonnesbeck C. Probabilistic programming in Python using PyMC3. <em>PeerJ Comput Sci</em>. 2016;2:e55. <a href="https://doi.org/10.7717/peerj-cs.55">doi:10.7717/peerj-cs.55</a></li>
<li id="r23">Carpenter B, Gelman A, Hoffman MD, et al. Stan: a probabilistic programming language. <em>J Stat Softw</em>. 2017;76(1):1-32. <a href="https://doi.org/10.18637/jss.v076.i01">doi:10.18637/jss.v076.i01</a></li>
<li id="r24">Gelman A. Prior distributions for variance parameters in hierarchical models. <em>Bayesian Anal</em>. 2006;1(3):515-534. <a href="https://doi.org/10.1214/06-BA117A">doi:10.1214/06-BA117A</a></li>
<li id="r25">Gelman A, Jakulin A, Pittau MG, Su YS. A weakly informative default prior distribution for logistic and other regression models. <em>Ann Appl Stat</em>. 2008;2(4):1360-1383. <a href="https://doi.org/10.1214/08-AOAS191">doi:10.1214/08-AOAS191</a></li>
<li id="r26">Greenland S, Senn SJ, Rothman KJ, et al. Statistical tests, P values, confidence intervals, and power: a guide to misinterpretations. <em>Eur J Epidemiol</em>. 2016;31:337-350. <a href="https://doi.org/10.1007/s10654-016-0149-3">doi:10.1007/s10654-016-0149-3</a></li>
<li id="r27">Morey RD, Hoekstra R, Rouder JN, Lee MD, Wagenmakers EJ. The fallacy of placing confidence in confidence intervals. <em>Psychon Bull Rev</em>. 2016;23:103-123. <a href="https://doi.org/10.3758/s13423-015-0947-8">doi:10.3758/s13423-015-0947-8</a></li>
<li id="r28">Kass RE, Raftery AE. Bayes factors. <em>J Am Stat Assoc</em>. 1995;90(430):773-795. <a href="https://doi.org/10.1080/01621459.1995.10476572">doi:10.1080/01621459.1995.10476572</a></li>
<li id="r29">Wagenmakers EJ. A practical solution to the pervasive problems of p values. <em>Psychon Bull Rev</em>. 2007;14(5):779-804. <a href="https://doi.org/10.3758/BF03194105">doi:10.3758/BF03194105</a></li>
<li id="r30">Kruschke JK, Liddell TM. The Bayesian new statistics: hypothesis testing, estimation, meta-analysis, and power analysis from a Bayesian perspective. <em>Psychon Bull Rev</em>. 2018;25:178-206. <a href="https://doi.org/10.3758/s13423-016-1221-4">doi:10.3758/s13423-016-1221-4</a></li>
<li id="r31">Wasserstein RL, Lazar NA. The ASA statement on p-values: context, process, and purpose. <em>Am Stat</em>. 2016;70(2):129-133. <a href="https://doi.org/10.1080/00031305.2016.1154108">doi:10.1080/00031305.2016.1154108</a></li>
<li id="r32">Bürkner PC. brms: an R package for Bayesian multilevel models using Stan. <em>J Stat Softw</em>. 2017;80(1):1-28. <a href="https://doi.org/10.18637/jss.v080.i01">doi:10.18637/jss.v080.i01</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>McElreath R. <em>Statistical Rethinking</em>. 2nd ed. CRC Press; 2020. — Chapters 1–4 and 13 cover exactly this unit, with the grid-approximation-first pedagogy it follows; the book's own code is R.</li>
<li>Gelman A, Carlin JB, Stern HS, Dunson DB, Vehtari A, Rubin DB. <em>Bayesian Data Analysis</em>. 3rd ed. CRC Press; 2013 (free PDF). — Chapters 1–5 for the theory, chapter 5 for the hierarchical model of §7.</li>
<li>Kruschke JK. <em>Doing Bayesian Data Analysis</em>. 2nd ed. Academic Press; 2015. — The gentlest full-length introduction, with extensive worked R code.</li>
<li>Gelman A, Hill J, Vehtari A. <em>Regression and Other Stories</em>. Cambridge University Press; 2020. — Bridges Unit 9 and this one; all examples in R.</li>
<li>Wood SN. <em>Core Statistics</em>. Cambridge University Press; 2015 (free PDF). — Chapter 6 derives Metropolis–Hastings and Gibbs sampling in R at the level of §6–§7.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>The <code>brms</code> documentation and vignettes (multilevel models, priors, <code>pp_check</code>): <a href="https://paul-buerkner.github.io/brms/">paul-buerkner.github.io/brms</a>.</li>
<li>The <code>lme4</code> vignette "Fitting Linear Mixed-Effects Models using lme4": <a href="https://cran.r-project.org/package=lme4">cran.r-project.org/package=lme4</a>.</li>
<li>Stan user's guide, "Efficiency tuning" and the funnel/non-centred discussion: <a href="https://mc-stan.org/docs/">mc-stan.org/docs</a>.</li>
<li>Statistical Rethinking lecture series and its R/Stan code: <a href="https://xcelab.net/rm/">xcelab.net/rm</a>.</li>
</ul>
